# Notebook 1: PMOS Endometrial DEG Analysis (GSE48301)
## PMOS-Cancer BridgeGraph — Phase 1 (Blueprint v2)

**Goal:** Identify differentially expressed genes in PMOS endometrium vs normal using limma with empirical-Bayes moderation. Output feeds into P(g) component of PCBS.

**Dataset:** GSE48301 — Piltonen et al. (2013)
- 6 PCOS + 6 Control women (BMI-matched)
- FACS-sorted into 4 cell types: Epithelial, Stromal, Mesenchymal, Endothelial
- Platform: Affymetrix HuGene 1.0 ST (GPL6244)
- 29 samples total

**Key changes from v1:**
- limma (empirical-Bayes) instead of Welch's t-test
- GEOparse for proper probe-to-gene mapping (v1 mapping was broken)
- Adjusted p-value < 0.05 only — no hard logFC cutoff
- All genes retained for downstream PCBS scoring

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q GEOparse gseapy rpy2 matplotlib-venn adjustText

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)

print('Setup complete.')

## 2. Download and Parse GSE48301 using GEOparse

GEOparse handles downloading the dataset AND the platform annotation (GPL6244) properly — this fixes the probe mapping bug from v1.

In [ ]:
import GEOparse

# Download GSE48301 (this downloads the series matrix + platform annotation)
print('Downloading GSE48301 from GEO (may take 1-2 minutes)...')
gse = GEOparse.get_GEO(geo='GSE48301', destdir='/tmp')

print(f'Dataset: {gse.metadata["title"][0]}')
print(f'Platform: {list(gse.gpls.keys())}')
print(f'Samples: {len(gse.gsms)}')

In [ ]:
# Extract sample metadata
metadata = []
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata['title'][0]
    
    # Determine condition
    condition = 'PCOS' if 'PCOS' in title else 'Control'
    
    # Determine cell type
    cell_type = 'Unknown'
    for ct in ['Epithelial', 'Stromal', 'Mesenchymal', 'Endothelial']:
        if ct in title:
            cell_type = ct
            break
    
    metadata.append({
        'GSM': gsm_name,
        'Title': title,
        'Condition': condition,
        'CellType': cell_type
    })

meta_df = pd.DataFrame(metadata).sort_values('GSM').reset_index(drop=True)

print('Sample metadata:')
print(meta_df[['GSM', 'Condition', 'CellType']].to_string())
print(f'\nCondition counts:')
print(meta_df['Condition'].value_counts().to_string())
print(f'\nCondition x CellType:')
print(pd.crosstab(meta_df['Condition'], meta_df['CellType']))

## 3. Extract Expression Matrix and Map Probes to Gene Symbols

In [ ]:
# Get the expression data from the first GSM to understand the structure
first_gsm = list(gse.gsms.values())[0]
print('Columns in GSM table:', list(first_gsm.table.columns))
print(f'Rows: {len(first_gsm.table)}')
first_gsm.table.head()

In [ ]:
# Build expression matrix from all GSMs
# Each GSM table has ID_REF and VALUE columns
expr_dict = {}
for gsm_name, gsm in gse.gsms.items():
    table = gsm.table
    expr_dict[gsm_name] = table.set_index('ID_REF')['VALUE'].astype(float)

expr = pd.DataFrame(expr_dict)
expr.index = expr.index.astype(str)

print(f'Expression matrix: {expr.shape[0]} probes x {expr.shape[1]} samples')
print(f'Value range: {expr.min().min():.2f} to {expr.max().max():.2f}')
print(f'Missing values: {expr.isnull().sum().sum()}')

# Drop rows with all NaN
expr = expr.dropna(how='all')
print(f'After removing empty rows: {expr.shape[0]} probes')

In [ ]:
# Get platform annotation for probe-to-gene mapping
gpl_name = list(gse.gpls.keys())[0]
gpl = gse.gpls[gpl_name]

print(f'Platform: {gpl_name}')
print(f'Annotation columns: {list(gpl.table.columns)}')

# Find the gene symbol column
gene_col = None
for col in gpl.table.columns:
    col_lower = col.lower()
    if 'gene_symbol' in col_lower or 'gene symbol' in col_lower or col_lower == 'symbol':
        gene_col = col
        break

# If not found, search more broadly
if gene_col is None:
    for col in gpl.table.columns:
        if 'gene' in col.lower() and 'symbol' in col.lower():
            gene_col = col
            break

# If still not found, try 'gene_assignment' and parse it
if gene_col is None:
    for col in gpl.table.columns:
        if 'gene_assignment' in col.lower():
            gene_col = col
            print(f'Found gene_assignment column: {col}')
            print('Will parse gene symbols from it.')
            break

print(f'Using column for gene mapping: {gene_col}')
print(f'\nSample values:')
print(gpl.table[['ID', gene_col]].dropna().head(10))

In [ ]:
# Create probe-to-gene mapping
annot = gpl.table[['ID', gene_col]].copy()
annot['ID'] = annot['ID'].astype(str)

# If the gene column is 'gene_assignment', parse out the gene symbol
# gene_assignment format: "NM_001005484 // OR4F5 // olfactory receptor... // 1p36.33 // 79501"
if 'assignment' in gene_col.lower():
    def extract_symbol(val):
        if pd.isna(val) or val == '---' or val == '':
            return None
        parts = str(val).split('//')
        if len(parts) >= 2:
            return parts[1].strip()
        return None
    annot['Gene'] = annot[gene_col].apply(extract_symbol)
else:
    annot['Gene'] = annot[gene_col]

# Clean up
annot = annot.dropna(subset=['Gene'])
annot = annot[annot['Gene'] != '---']
annot = annot[annot['Gene'] != '']

probe_to_gene = dict(zip(annot['ID'], annot['Gene']))
print(f'Probe-to-gene mapping: {len(probe_to_gene)} probes mapped')

# Map probes to genes
expr['Gene'] = expr.index.map(probe_to_gene)

n_before = len(expr)
expr = expr.dropna(subset=['Gene'])
print(f'Mapped: {len(expr)}/{n_before} probes have gene symbols')

# For duplicate genes, keep probe with highest mean expression
sample_cols = [c for c in expr.columns if c != 'Gene']
expr['mean_expr'] = expr[sample_cols].mean(axis=1)
expr = expr.sort_values('mean_expr', ascending=False)
expr = expr.drop_duplicates(subset='Gene', keep='first')
expr = expr.drop('mean_expr', axis=1)
expr = expr.set_index('Gene')

print(f'Final: {expr.shape[0]} unique genes x {expr.shape[1]} samples')

# Verify known genes are present
check_genes = ['ESR1', 'PTEN', 'PIK3CA', 'AKT1', 'INSR', 'AR', 'PGR', 'MMP9', 'JUN']
found = [g for g in check_genes if g in expr.index]
print(f'\nKnown gene check: {len(found)}/{len(check_genes)} found')
print(f'  Found: {found}')
print(f'  Missing: {[g for g in check_genes if g not in expr.index]}')

## 4. Quality Control

In [ ]:
# Distribution of expression values per sample
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot per sample
expr_plot = expr[meta_df['GSM'].tolist()]
colors = ['#e74c3c' if c == 'PCOS' else '#3498db' for c in meta_df['Condition']]
bp = axes[0].boxplot([expr_plot[col].values for col in expr_plot.columns],
                      patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[0].set_xticklabels(meta_df['GSM'].str.replace('GSM', ''), rotation=90, fontsize=6)
axes[0].set_ylabel('Expression (log2)')
axes[0].set_title('Expression distribution per sample (red=PCOS, blue=Control)')

# Density plot
for i, col in enumerate(expr_plot.columns):
    color = '#e74c3c' if meta_df.iloc[i]['Condition'] == 'PCOS' else '#3498db'
    axes[1].hist(expr_plot[col].values, bins=50, alpha=0.15, color=color, density=True)
axes[1].set_xlabel('Expression (log2)')
axes[1].set_ylabel('Density')
axes[1].set_title('Expression density (red=PCOS, blue=Control)')

plt.tight_layout()
plt.savefig('figures/01_qc_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = expr.T.values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=5)
pcs = pca.fit_transform(X_scaled)

# Align PCA results with metadata
sample_order = expr.columns.tolist()
meta_aligned = meta_df.set_index('GSM').loc[sample_order].reset_index()

pca_df = pd.DataFrame({
    'PC1': pcs[:, 0], 'PC2': pcs[:, 1],
    'Condition': meta_aligned['Condition'].values,
    'CellType': meta_aligned['CellType'].values
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By Condition
for cond, color in [('PCOS', '#e74c3c'), ('Control', '#3498db')]:
    mask = pca_df['Condition'] == cond
    axes[0].scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
                    c=color, label=cond, s=80, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA by Condition')
axes[0].legend()

# By Cell Type
ct_colors = {'Epithelial': '#e74c3c', 'Stromal': '#3498db',
             'Mesenchymal': '#2ecc71', 'Endothelial': '#f39c12'}
for ct, color in ct_colors.items():
    mask = pca_df['CellType'] == ct
    axes[1].scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
                    c=color, label=ct, s=80, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('PCA by Cell Type')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/02_pca_gse48301.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Variance explained: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, '
      f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%, '
      f'PC3={pca.explained_variance_ratio_[2]*100:.1f}%')

## 5. Differential Expression Analysis using limma

We use limma with empirical-Bayes moderation instead of Welch's t-test. This is the gold standard for microarray data, especially with small sample sizes. It borrows information across all genes to stabilize variance estimates.

**Filtering:** Adjusted p-value (BH) < 0.05 only. No hard logFC cutoff — the continuous logFC value feeds into PCBS P(g) component downstream.

In [ ]:
# Setup R environment for limma
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, r
from rpy2.robjects.packages import importr

pandas2ri.activate()

# Install limma if needed
r('''
if (!requireNamespace("limma", quietly = TRUE)) {
    if (!requireNamespace("BiocManager", quietly = TRUE))
        install.packages("BiocManager")
    BiocManager::install("limma")
}
library(limma)
''')

print('limma loaded successfully in R.')

In [ ]:
def run_limma(expr_df, group1_samples, group2_samples,
              group1_name='PCOS', group2_name='Control'):
    """
    Run limma DEG analysis via rpy2.
    Returns DataFrame with Gene, logFC, AveExpr, t, P.Value, adj.P.Val, B.
    """
    # Prepare expression matrix (genes x samples in correct order)
    all_samples = group1_samples + group2_samples
    expr_sub = expr_df[all_samples].copy()
    
    # Create design labels
    groups = [group1_name] * len(group1_samples) + [group2_name] * len(group2_samples)
    
    # Transfer to R
    r_expr = pandas2ri.py2rpy(expr_sub)
    ro.globalenv['expr_matrix'] = r_expr
    ro.globalenv['groups'] = ro.StrVector(groups)
    ro.globalenv['g1_name'] = group1_name
    ro.globalenv['g2_name'] = group2_name
    
    # Run limma in R
    r('''
    # Create design matrix
    groups_factor <- factor(groups, levels = c(g2_name, g1_name))
    design <- model.matrix(~ groups_factor)
    colnames(design) <- c("Intercept", paste0(g1_name, "_vs_", g2_name))
    
    # Fit linear model
    fit <- lmFit(as.matrix(expr_matrix), design)
    
    # Empirical Bayes moderation
    fit <- eBayes(fit)
    
    # Get results for the contrast (group1 vs group2)
    results <- topTable(fit, coef = 2, number = Inf, sort.by = "none",
                        adjust.method = "BH")
    ''')
    
    # Get results back to Python
    r_results = ro.globalenv['results']
    results_df = pandas2ri.rpy2py(r_results)
    
    # Add gene names
    results_df['Gene'] = expr_sub.index.tolist()
    
    # Rename columns for consistency
    results_df = results_df.rename(columns={
        'P.Value': 'pvalue',
        'adj.P.Val': 'adj_pvalue'
    })
    
    # Assign direction based on adj_pvalue < 0.05 only (no logFC cutoff)
    results_df['Direction'] = 'NS'
    results_df.loc[(results_df['adj_pvalue'] < 0.05) & (results_df['logFC'] > 0), 'Direction'] = 'Up'
    results_df.loc[(results_df['adj_pvalue'] < 0.05) & (results_df['logFC'] < 0), 'Direction'] = 'Down'
    
    results_df = results_df.sort_values('pvalue')
    
    n_up = (results_df['Direction'] == 'Up').sum()
    n_down = (results_df['Direction'] == 'Down').sum()
    n_total = n_up + n_down
    print(f'  DEGs (adj.p < 0.05): {n_up} up + {n_down} down = {n_total} total')
    print(f'  Total genes tested: {len(results_df)}')
    
    return results_df

print('limma DEG function ready.')

In [ ]:
# Run overall DEG analysis: all PCOS vs all Control
pcos_samples = meta_df[meta_df['Condition'] == 'PCOS']['GSM'].tolist()
ctrl_samples = meta_df[meta_df['Condition'] == 'Control']['GSM'].tolist()

print(f'=== Overall Analysis: {len(pcos_samples)} PCOS vs {len(ctrl_samples)} Control ===')
deg_overall = run_limma(expr, pcos_samples, ctrl_samples)
deg_overall.to_csv('results/deg_overall_gse48301.csv', index=False)

print(f'\nlogFC distribution of significant DEGs:')
sig = deg_overall[deg_overall['Direction'] != 'NS']
if len(sig) > 0:
    print(f'  Mean |logFC|: {sig["logFC"].abs().mean():.3f}')
    print(f'  Median |logFC|: {sig["logFC"].abs().median():.3f}')
    print(f'  Max |logFC|: {sig["logFC"].abs().max():.3f}')
    print(f'  Genes with |logFC| > 1: {(sig["logFC"].abs() > 1).sum()}')
    print(f'  Genes with |logFC| > 0.5: {(sig["logFC"].abs() > 0.5).sum()}')
    print(f'  Genes with |logFC| < 0.3: {(sig["logFC"].abs() < 0.3).sum()}')
else:
    print('  No significant DEGs found.')

In [ ]:
# Run per-cell-type DEG analysis (supplementary — cell-type-specific evidence)
deg_by_celltype = {}

for ct in ['Epithelial', 'Stromal', 'Mesenchymal', 'Endothelial']:
    ct_meta = meta_df[meta_df['CellType'] == ct]
    ct_pcos = ct_meta[ct_meta['Condition'] == 'PCOS']['GSM'].tolist()
    ct_ctrl = ct_meta[ct_meta['Condition'] == 'Control']['GSM'].tolist()
    
    if len(ct_pcos) < 2 or len(ct_ctrl) < 2:
        print(f'\n{ct}: Skipping (only {len(ct_pcos)} PCOS, {len(ct_ctrl)} Control)')
        continue
    
    print(f'\n=== {ct}: {len(ct_pcos)} PCOS vs {len(ct_ctrl)} Control ===')
    deg_ct = run_limma(expr, ct_pcos, ct_ctrl)
    deg_by_celltype[ct] = deg_ct
    deg_ct.to_csv(f'results/deg_{ct.lower()}_gse48301.csv', index=False)

print('\nAll DEG analyses complete.')

## 6. Volcano Plots

In [ ]:
def plot_volcano(deg_df, title, ax, top_n=10):
    """Volcano plot with top gene labels."""
    colors = {'Up': '#e74c3c', 'Down': '#3498db', 'NS': '#cccccc'}
    
    for direction in ['NS', 'Up', 'Down']:
        mask = deg_df['Direction'] == direction
        ax.scatter(deg_df.loc[mask, 'logFC'],
                   -np.log10(deg_df.loc[mask, 'adj_pvalue'].clip(lower=1e-50)),
                   c=colors[direction], s=8, alpha=0.5, label=direction)
    
    # Label top significant genes
    sig = deg_df[deg_df['Direction'] != 'NS'].sort_values('adj_pvalue').head(top_n)
    for _, row in sig.iterrows():
        ax.annotate(row['Gene'],
                    (row['logFC'], -np.log10(max(row['adj_pvalue'], 1e-50))),
                    fontsize=6, alpha=0.8, ha='center')
    
    ax.axhline(-np.log10(0.05), color='grey', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_xlabel('log2 Fold Change')
    ax.set_ylabel('-log10(adj. p-value)')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7, loc='upper right')


# Plot volcanos
cell_types_with_data = list(deg_by_celltype.keys())
n_plots = 1 + len(cell_types_with_data)

fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4.5))
if n_plots == 1:
    axes = [axes]

plot_volcano(deg_overall, 'Overall (PCOS vs Control)', axes[0])
for i, ct in enumerate(cell_types_with_data):
    plot_volcano(deg_by_celltype[ct], f'{ct}', axes[i + 1])

plt.tight_layout()
plt.savefig('figures/03_volcanos_gse48301.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Pathway Enrichment (GO + KEGG)

In [ ]:
import gseapy as gp

def run_enrichment(gene_list, name, max_genes=300):
    """Run GO and KEGG enrichment on a gene list."""
    genes = [str(g) for g in gene_list[:max_genes]]
    
    if len(genes) < 5:
        print(f'  {name}: Only {len(genes)} genes, skipping.')
        return None
    
    print(f'  {name}: Enriching {len(genes)} genes...')
    
    try:
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=['GO_Biological_Process_2023', 'KEGG_2021_Human'],
            organism='human',
            outdir=None,
            no_plot=True
        )
        results = enr.results
        sig = results[results['Adjusted P-value'] < 0.05].sort_values('Adjusted P-value')
        sig.to_csv(f'results/enrichment_{name}.csv', index=False)
        
        n_go = len(sig[sig['Gene_set'] == 'GO_Biological_Process_2023'])
        n_kegg = len(sig[sig['Gene_set'] == 'KEGG_2021_Human'])
        print(f'    Significant: {n_go} GO terms, {n_kegg} KEGG pathways')
        return sig
    except Exception as e:
        print(f'    Enrichment error: {e}')
        return None


# Overall DEGs — upregulated
print('=== Enrichment: Overall Upregulated ===')
up_genes = deg_overall[deg_overall['Direction'] == 'Up'].sort_values('adj_pvalue')['Gene'].tolist()
enr_up = run_enrichment(up_genes, 'overall_up')

# Overall DEGs — downregulated
print('\n=== Enrichment: Overall Downregulated ===')
down_genes = deg_overall[deg_overall['Direction'] == 'Down'].sort_values('adj_pvalue')['Gene'].tolist()
enr_down = run_enrichment(down_genes, 'overall_down')

# Per cell type (all DEGs)
enr_by_celltype = {}
for ct in cell_types_with_data:
    print(f'\n=== Enrichment: {ct} ===')
    ct_genes = deg_by_celltype[ct][deg_by_celltype[ct]['Direction'] != 'NS'].sort_values('adj_pvalue')['Gene'].tolist()
    enr_ct = run_enrichment(ct_genes, ct.lower())
    if enr_ct is not None:
        enr_by_celltype[ct] = enr_ct

print('\nAll enrichments complete.')

In [ ]:
# Plot top enriched terms
def plot_top_terms(enr_df, title, gene_set_filter=None, top_n=15):
    if enr_df is None or len(enr_df) == 0:
        print(f'No significant terms for: {title}')
        return
    
    df = enr_df.copy()
    if gene_set_filter:
        df = df[df['Gene_set'] == gene_set_filter]
    
    df = df.head(top_n)
    if len(df) == 0:
        return
    
    df['Term_short'] = df['Term'].str[:55]
    
    fig, ax = plt.subplots(figsize=(10, max(3, len(df) * 0.35)))
    ax.barh(range(len(df)), -np.log10(df['Adjusted P-value']),
            color='#2ecc71', edgecolor='black', linewidth=0.3)
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df['Term_short'], fontsize=8)
    ax.set_xlabel('-log10(Adjusted P-value)')
    ax.set_title(title)
    ax.invert_yaxis()
    plt.tight_layout()
    safe_title = title.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
    plt.savefig(f'figures/04_enrich_{safe_title}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_top_terms(enr_up, 'Overall Upregulated — GO BP', 'GO_Biological_Process_2023')
plot_top_terms(enr_up, 'Overall Upregulated — KEGG', 'KEGG_2021_Human')
plot_top_terms(enr_down, 'Overall Downregulated — GO BP', 'GO_Biological_Process_2023')
plot_top_terms(enr_down, 'Overall Downregulated — KEGG', 'KEGG_2021_Human')

## 8. Validate Known PMOS-Cancer Genes

In [ ]:
# Check known PMOS-EC genes from published literature
known_genes = {
    'IL10': 'Anti-inflammatory cytokine (Ye et al.)',
    'CXCL8': 'Chemokine IL-8 (Ye et al.)',
    'IFNG': 'Interferon gamma (Ye et al.)',
    'MMP9': 'Matrix metalloproteinase (Ye et al.)',
    'PECAM1': 'Endothelial adhesion (Ye et al.)',
    'MYD88': 'Innate immune signaling (Ye et al.)',
    'ESR1': 'Estrogen receptor',
    'JUN': 'AP-1 transcription factor',
    'PIK3CA': 'PI3K catalytic subunit',
    'AKT1': 'PI3K/AKT kinase',
    'PTEN': 'Tumor suppressor',
    'INSR': 'Insulin receptor',
    'IGF1R': 'IGF-1 receptor',
    'AR': 'Androgen receptor',
    'PGR': 'Progesterone receptor',
    'CYP19A1': 'Aromatase',
    'SERPINE1': 'PAI-1',
    'TP53': 'Tumor suppressor p53',
    'CTNNB1': 'Beta-catenin',
    'KRAS': 'RAS oncogene',
}

print('Known PMOS-EC gene status in GSE48301 (limma):')
print('=' * 85)
print(f'{"Gene":<10} {"Direction":<8} {"logFC":>8} {"adj.p":>10} {"Description"}')
print('-' * 85)

check_results = []
for gene, desc in known_genes.items():
    row = deg_overall[deg_overall['Gene'] == gene]
    if len(row) > 0:
        r = row.iloc[0]
        direction = r['Direction']
        logfc = r['logFC']
        adjp = r['adj_pvalue']
        status = f'{direction:<8} {logfc:>8.3f} {adjp:>10.4f}'
    else:
        status = 'Not in dataset'
        direction = 'NA'
        logfc = None
        adjp = None
    
    print(f'{gene:<10} {status} {desc}')
    check_results.append({'Gene': gene, 'Description': desc, 'Direction': direction,
                          'logFC': logfc, 'adj_pvalue': adjp})

check_df = pd.DataFrame(check_results)
check_df.to_csv('results/known_gene_check_gse48301.csv', index=False)

found_sig = check_df[(check_df['Direction'] == 'Up') | (check_df['Direction'] == 'Down')]
found_ns = check_df[check_df['Direction'] == 'NS']
found_na = check_df[check_df['Direction'] == 'NA']
print(f'\nSummary: {len(found_sig)} significant, {len(found_ns)} not significant, {len(found_na)} not in dataset')

## 9. Cell-Type-Specific DEG Summary

In [ ]:
# Summary of DEGs per cell type
all_sig_genes = {}
for ct in cell_types_with_data:
    sig = deg_by_celltype[ct][deg_by_celltype[ct]['Direction'] != 'NS']
    all_sig_genes[ct] = set(sig['Gene'].tolist())
    print(f'{ct}: {len(all_sig_genes[ct])} DEGs')

# Overlap between cell types
from itertools import combinations

print('\n--- Shared DEGs ---')
for ct1, ct2 in combinations(all_sig_genes.keys(), 2):
    shared = all_sig_genes[ct1] & all_sig_genes[ct2]
    print(f'  {ct1} & {ct2}: {len(shared)}')

# Genes in ALL cell types
if len(all_sig_genes) >= 2:
    all_ct_genes = set.intersection(*all_sig_genes.values())
    print(f'\n  In ALL cell types: {len(all_ct_genes)}')
    if 0 < len(all_ct_genes) <= 30:
        print(f'    {sorted([str(g) for g in all_ct_genes])}')

# Save cell-type DEG summary
celltype_summary = []
for ct, genes in all_sig_genes.items():
    for g in genes:
        row = deg_by_celltype[ct][deg_by_celltype[ct]['Gene'] == g].iloc[0]
        celltype_summary.append({
            'Gene': g, 'CellType': ct,
            'Direction': row['Direction'], 'logFC': row['logFC'],
            'adj_pvalue': row['adj_pvalue']
        })

celltype_df = pd.DataFrame(celltype_summary)
celltype_df.to_csv('results/celltype_deg_summary_gse48301.csv', index=False)
print('\nCell-type DEG summary saved.')

## 10. Save All Outputs for Downstream Notebooks

In [ ]:
# Save expression matrix (gene-level)
expr.to_csv('results/expr_matrix_gse48301.csv')

# Save ALL genes with their limma statistics (for PCBS P(g) component)
# This is the key output — PCBS needs continuous logFC and p-values, not just DEG labels
deg_overall[['Gene', 'logFC', 'AveExpr', 't', 'pvalue', 'adj_pvalue', 'B', 'Direction']].to_csv(
    'results/all_genes_limma_gse48301.csv', index=False
)

# Save as pickle for easy loading
import pickle
with open('results/gse48301_outputs.pkl', 'wb') as f:
    pickle.dump({
        'deg_overall': deg_overall,
        'deg_by_celltype': deg_by_celltype,
        'meta_df': meta_df,
        'celltype_summary': celltype_df,
        'enr_up': enr_up,
        'enr_down': enr_down,
    }, f)

# Final Summary
print('=' * 60)
print('NOTEBOOK 1 COMPLETE — GSE48301 PMOS Endometrial Analysis')
print('=' * 60)
print(f'\nDataset: GSE48301 (Piltonen et al., 2013)')
print(f'Platform: GPL6244 (Affymetrix HuGene 1.0 ST)')
print(f'Samples: {len(meta_df)} ({len(pcos_samples)} PCOS, {len(ctrl_samples)} Control)')
print(f'Genes analyzed: {expr.shape[0]}')
print(f'\nMethod: limma (empirical-Bayes moderation)')
print(f'Filter: adj.p < 0.05, no logFC cutoff')
print(f'\nOverall DEGs: {(deg_overall["Direction"] != "NS").sum()}')
print(f'  Upregulated: {(deg_overall["Direction"] == "Up").sum()}')
print(f'  Downregulated: {(deg_overall["Direction"] == "Down").sum()}')
for ct in cell_types_with_data:
    n = (deg_by_celltype[ct]['Direction'] != 'NS').sum()
    print(f'  {ct}: {n} DEGs')
print(f'\nKey output for PCBS: results/all_genes_limma_gse48301.csv')
print(f'  (Contains ALL {len(deg_overall)} genes with continuous logFC + adj.p-value)')
print(f'\n-> Next: Notebook 2 (Cancer DEG analysis — TCGA-UCEC)')

In [ ]:
# Download results
!zip -r gse48301_results_v2.zip results/ figures/
from google.colab import files
files.download('gse48301_results_v2.zip')
print('Results downloaded.')